# Part A3 & A4 - Evaluating the Campaign, and Our Recommendations

**What this notebook does:** checks whether the growth team's claim about `CAMP_WA_002` holds up (**A3**), and gives our three ranked recommendations with the working shown (**A4**). Uses the exact same rulebook (`common.py`) as A1 and A2, so nothing here can quietly disagree.

### Step 0 - Load the data, same rules as A1 and A2
Same shared rulebook every time, so nothing here can quietly disagree with the other two notebooks.

In [1]:
import pandas as pd
import common

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

DATA_DIR = common.find_data_dir()
captains, doc_events, approvals, activation, nudges = common.load_all(DATA_DIR)
master, master_mature, master_recent = common.build_master(captains, approvals, activation)

master_mature = master_mature.copy()
print(f"master_mature: {len(master_mature):,} rows")
print(nudges["campaign_id"].value_counts())

master_mature: 20,207 rows
campaign_id
CAMP_WA_002      8673
CAMP_WA_001      3807
CAMP_SMS_004     2661
CAMP_CALL_009    1173
Name: count, dtype: int64


### Does this campaign get sent the same way as the others?
Checks when `CAMP_WA_002` is actually sent, compared to every other campaign. First step in checking whether growth's "big win" claim **survives being poked**, as the brief asks.

In [2]:

m = nudges.merge(captains[["captain_id","signup_ts"]], on="captain_id")
m["days_since_signup"] = (m["sent_ts"] - m["signup_ts"]).dt.total_seconds()/86400
print("Days since signup when each campaign is sent:")
print(m.groupby("campaign_id")["days_since_signup"].agg(["mean","median","count"]).round(2))

Days since signup when each campaign is sent:
               mean  median  count
campaign_id                       
CAMP_CALL_009  1.00    0.88   1173
CAMP_SMS_004   1.00    0.90   2661
CAMP_WA_001    1.01    0.90   3807
CAMP_WA_002    2.36    2.23   8673


**Finding:** this campaign is sent over twice as late as every other one - about 2.2 days after signup, versus under a day for the rest. Anyone who already quit before then could never have received it.

### Who actually receives this campaign?
Checks whether the people who get this message were already struggling, or already doing fine. This tells us whether the comparison is even fair.

In [3]:
fails = doc_events[doc_events["event_type"]=="verification_fail"]
wa2_sent = nudges[nudges["campaign_id"]=="CAMP_WA_002"][["captain_id","sent_ts"]]
chk = wa2_sent.merge(fails[["captain_id","event_ts"]], on="captain_id", how="left")
chk["fail_before_nudge"] = chk["event_ts"] < chk["sent_ts"]
had_fail_before = chk.groupby("captain_id")["fail_before_nudge"].any()

any_fail_pop = doc_events[doc_events["event_type"]=="verification_fail"]["captain_id"].nunique()

print(f"CAMP_WA_002 recipients with a failure BEFORE the nudge: {had_fail_before.mean():.1%}")
print(f"Overall captain population with any failure ever:       {any_fail_pop/len(captains):.1%}")

CAMP_WA_002 recipients with a failure BEFORE the nudge: 29.4%
Overall captain population with any failure ever:       55.7%


**Confirms it:** people who get this message already have a *lower* chance of having failed something than the general population - the opposite of what you'd expect if it targeted strugglers. It looks like it's just sent to whoever's still around, which flatters the result.

### The claim, as growth would present it
The plain, no-adjustment comparison: people who got the message vs everyone else. This is almost certainly the number behind the 5x budget request.

In [4]:
wa2_ids = set(nudges[nudges["campaign_id"]=="CAMP_WA_002"]["captain_id"])
master_mature["got_wa2"] = master_mature["captain_id"].isin(wa2_ids)
master_mature["approved_flag"] = master_mature["final_status"] == "approved"

naive = master_mature.groupby("got_wa2")["approved_flag"].agg(["mean","count"])
naive_lift = naive.loc[True,"mean"] - naive.loc[False,"mean"]
print(naive)
print(f"\nNAIVE LIFT: {naive_lift:.1%}  (this is the number behind the 'big win' claim)")

             mean  count
got_wa2                 
False    0.112088  13079
True     0.289843   7128

NAIVE LIFT: 17.8%  (this is the number behind the 'big win' claim)


### The fair comparison - this is the number we'd defend
Only compares people who were still "in the running" at the same point in time, removing the unfair head start the claim above has. This is the **difference vs. effect** check the brief specifically asks for.

In [5]:
LANDMARK_DAYS = m[m["campaign_id"]=="CAMP_WA_002"]["days_since_signup"].median()
print(f"Landmark: {LANDMARK_DAYS:.2f} days post-signup (median CAMP_WA_002 send time)")

last_activity = doc_events.groupby("captain_id")["event_ts"].max().rename("last_activity_ts")
mm = master_mature.merge(last_activity, on="captain_id", how="left")
mm["last_activity_ts"] = mm["last_activity_ts"].fillna(mm["signup_ts"])
mm["days_active"] = (mm["last_activity_ts"] - mm["signup_ts"]).dt.total_seconds()/86400
mm["survived_to_landmark"] = mm["days_active"] >= LANDMARK_DAYS

print()
print("Share of each group that even survives to the landmark:")
print(mm.groupby("got_wa2")["survived_to_landmark"].agg(["mean","sum","count"]))

Landmark: 2.23 days post-signup (median CAMP_WA_002 send time)

Share of each group that even survives to the landmark:
             mean   sum  count
got_wa2                       
False    0.479318  6269  13079
True     0.900393  6418   7128


### Reading the fair comparison
With the unfair head start removed, this shows the real, defensible size of the campaign's benefit.

In [6]:
landmark_pop = mm[mm["survived_to_landmark"]]
print(f"Landmark-matched population: {len(landmark_pop):,} of {len(mm):,} mature captains")

corrected = landmark_pop.groupby("got_wa2")["approved_flag"].agg(["mean","count"])
corrected_lift = corrected.loc[True,"mean"] - corrected.loc[False,"mean"]
print(corrected)
print(f"\nLANDMARK-CORRECTED LIFT: {corrected_lift:.1%}")
print(f"(vs. naive lift of {naive_lift:.1%} -- roughly half the size once survival bias is controlled for)")

Landmark-matched population: 12,687 of 20,207 mature captains
             mean  count
got_wa2                 
False    0.233690   6269
True     0.321596   6418

LANDMARK-CORRECTED LIFT: 8.8%
(vs. naive lift of 17.8% -- roughly half the size once survival bias is controlled for)


### Double-checking the fair comparison holds up
Makes sure this result isn't secretly just re-showing the phone-quality or channel story from A2 in disguise.

In [7]:
print("device_tier mix within landmark population, by got_wa2:")
print(pd.crosstab(landmark_pop["got_wa2"], landmark_pop["device_tier"], normalize="index").round(3))
print()
print("acquisition_channel mix within landmark population, by got_wa2:")
print(pd.crosstab(landmark_pop["got_wa2"], landmark_pop["acquisition_channel"], normalize="index").round(3))

device_tier mix within landmark population, by got_wa2:
device_tier   high    low    mid
got_wa2                         
False        0.156  0.463  0.381
True         0.164  0.431  0.405

acquisition_channel mix within landmark population, by got_wa2:
acquisition_channel  fos_field  gc_telecalling  organic_app  paid_digital  referral
got_wa2                                                                            
False                    0.292           0.052        0.327         0.115     0.215
True                     0.298           0.057        0.314         0.118     0.213


**Balanced.** Phone type and sign-up channel look the same in both groups here — so this result isn't just re-showing an A2 finding in disguise.

### One more check nobody asked for
Checks whether people in the "no campaign" group actually got a *different* message instead, which could quietly bias the result. The brief rewards noticing things it didn't ask about.

In [8]:
control_pop = landmark_pop[~landmark_pop["got_wa2"]].copy()
other_campaign_ids = set(
    nudges[(nudges["captain_id"].isin(control_pop["captain_id"])) & (nudges["campaign_id"] != "CAMP_WA_002")]["captain_id"]
)
control_pop["got_other_nudge"] = control_pop["captain_id"].isin(other_campaign_ids)

print(f"Control group size (landmark-survived, no WA2): {len(control_pop):,}")
print(f"Of those, received a different campaign: {control_pop['got_other_nudge'].sum():,}")
print()
print("Approval rate within control, split by other-nudge exposure:")
print(control_pop.groupby("got_other_nudge")["approved_flag"].agg(["mean", "count"]))

rate_gap = (
    control_pop[control_pop["got_other_nudge"]]["approved_flag"].mean()
    - control_pop[~control_pop["got_other_nudge"]]["approved_flag"].mean()
)
print(f"\nGap: {rate_gap:.1%} points -- small relative to the 8.8pt WA2 lift being evaluated.")
print("Contamination exists but doesn't meaningfully change the A3 conclusion.")

Control group size (landmark-survived, no WA2): 6,269
Of those, received a different campaign: 1,981

Approval rate within control, split by other-nudge exposure:
                     mean  count
got_other_nudge                 
False            0.232509   4288
True             0.236244   1981

Gap: 0.4% points -- small relative to the 8.8pt WA2 lift being evaluated.
Contamination exists but doesn't meaningfully change the A3 conclusion.


**Small effect, doesn't change our answer.** Some of the "no campaign" group did get a different message, but the effect on the numbers is minor.

### Turning the fair number into 'per month'
Same current sign-up volume rule as A2. Converts the percentage-point gap into an actual number of drivers per month - at today's scale, and at the requested 5x scale.

In [9]:

# Consistent with Rec 1's methodology: share of a typical mature-cohort
# signup, scaled by the CURRENT signup run-rate -- not a flat historical
# month divisor (an earlier draft hardcoded /6, later found to be /5 too,
# since the mature cohort spans 5 months not 6, and never updated to match
# Rec 1's current-run-rate basis).
wa2_reached_landmark = landmark_pop["got_wa2"].sum()
HISTORICAL_AVG, CURRENT_RUNRATE = common.monthly_signup_volume(captains)
TOTAL_MATURE = len(master_mature)
monthly_reach_current = wa2_reached_landmark / TOTAL_MATURE * CURRENT_RUNRATE

print(f"CAMP_WA_002 recipients within the landmark-matched population: {wa2_reached_landmark:,}")
print(f"-> current monthly reach: ~{monthly_reach_current:.0f} captains/month")
print()
print("=== At CURRENT scale ===")
print(f"If the naive (uncorrected) lift were real:      ~{monthly_reach_current*naive_lift:.0f} incremental approvals/month")
print(f"Landmark-corrected, defensible estimate:          ~{monthly_reach_current*corrected_lift:.0f} incremental approvals/month")
print()
print("=== IF scaled 5x, as requested (optimistic -- assumes same lift on a 5x larger, newly-reached population) ===")
print(f"Naive-claim basis:               ~{monthly_reach_current*5*naive_lift:.0f} incremental approvals/month")
print(f"Landmark-corrected basis:         ~{monthly_reach_current*5*corrected_lift:.0f} incremental approvals/month")

CAMP_WA_002 recipients within the landmark-matched population: 6,418
-> current monthly reach: ~1522 captains/month

=== At CURRENT scale ===
If the naive (uncorrected) lift were real:      ~271 incremental approvals/month
Landmark-corrected, defensible estimate:          ~134 incremental approvals/month

=== IF scaled 5x, as requested (optimistic -- assumes same lift on a 5x larger, newly-reached population) ===
Naive-claim basis:               ~1353 incremental approvals/month
Landmark-corrected basis:         ~669 incremental approvals/month


### A3 conclusion - the number we'd put in a deck

**The number:** roughly **8.8 percentage points** better approval rate — about **~134 more approved drivers a month** right now. That's about half of the ~271/month the naive comparison claims.

**How sure are we?** Fairly sure, not certain. We removed the biggest unfair advantage in the comparison and double-checked it isn't hiding another cause. But this still isn't a proper side-by-side test.

**On the 5x budget:** don't approve it yet. Run a real, fair, small test first — send it to half of eligible people, not the other half, and compare. Then we'll know the real number before spending real money.


# A4 — Recommendations, ranked

Three recommendations, each grounded in a number already verified in
A1, A2, or A3 above -- nothing here is a fresh claim.


### The math behind Recommendation 1
Shows exactly how the phone-quality fix's monthly number is worked out - the brief says don't ship a number you can't defend line by line, so here's the line-by-line version.

In [10]:
# RC first-attempt fail rate by device tier, and the resulting impact estimate --
# computed live via common.py, using the SAME current-run-rate basis as A2
# (not the historical average -- see A2's stated-assumption cell for why).

doc_events_mature = doc_events[doc_events["captain_id"].isin(master_mature["captain_id"])]
fail_rate, n_by_tier, rc_attempt_rate = common.rc_fail_rate_by_device_tier(doc_events_mature, master_mature)
print("RC first-attempt fail rate by device tier:")
print(fail_rate.round(3))

HISTORICAL_AVG, CURRENT_RUNRATE = common.monthly_signup_volume(captains)
TOTAL_MATURE = len(master_mature)

# n_by_tier[tier] / TOTAL_MATURE = share of a typical signup cohort that is
# tier-X AND an RC attempter. Multiplying by CURRENT_RUNRATE (not TOTAL_MATURE
# signups, and not "share among RC-attempters" x total signups -- that
# conflation was a bug in an earlier draft) gives the correct monthly count.
low_monthly = n_by_tier["low"] * CURRENT_RUNRATE / TOTAL_MATURE
mid_monthly = n_by_tier["mid"] * CURRENT_RUNRATE / TOTAL_MATURE
excess_fails_per_month = low_monthly * (fail_rate["low"] - fail_rate["high"]) + mid_monthly * (fail_rate["mid"] - fail_rate["high"])

rc = doc_events_mature[doc_events_mature["doc_type"] == "RC"]
rc_retry = common.post_failure_outcome(doc_events_mature, "RC")
never_retry_rate = rc_retry["never_retried_pct"]
rc_cleared_ids = set(rc[rc["event_type"] == "verification_pass"]["captain_id"])
conv_rate = (master_mature[master_mature["captain_id"].isin(rc_cleared_ids)]["final_status"] == "approved").mean()
net_new_approvals_per_month = excess_fails_per_month * never_retry_rate * conv_rate

print(f"\nExcess RC first-attempt failures/month (current run-rate basis): {excess_fails_per_month:.0f}")
print(f"RC never-retry-after-fail rate: {never_retry_rate:.1%}")
print(f"RC-cleared -> approved conversion rate: {conv_rate:.1%}")
print(f"\nNET NEW APPROVED CAPTAINS/MONTH: {net_new_approvals_per_month:.0f}")

RC first-attempt fail rate by device tier:
device_tier
high    0.215
low     0.438
mid     0.292
dtype: float64

Excess RC first-attempt failures/month (current run-rate basis): 494
RC never-retry-after-fail rate: 49.9%
RC-cleared -> approved conversion rate: 27.2%

NET NEW APPROVED CAPTAINS/MONTH: 67


## Recommendation 1 - Fix the photo check on the Registration Certificate

**What to do:** Catch a blurry photo the moment someone tries to upload it, and ask for a retake right away - instead of finding out days later.

**Why:** This document fails more than any other, and half the failures are just bad photos - twice as common on cheaper phones. Fixing this is worth roughly **~67 more approved drivers a month**.

**Cost / risk:** Moderate effort to build, low risk - it's a better camera prompt, not a policy change.

**How we'd know it worked:** Check the failure rate by phone type again after 4 weeks. It should drop and match good phones' rate.


## Recommendation 2 - Send a reminder to people doing fine who went quiet

**What to do:** A friendly reminder, 1–2 days after someone goes inactive, for anyone who's passed every check so far but just stopped. This is a different message from a "please retry" nudge - nothing went wrong for them, they just need a nudge to continue.

**Why:** This is our single largest untapped group - a third of everyone we lose - and today we do nothing about it.

**Cost / risk:** Low cost, low risk. We'd test it on a small group first, so we know the real impact before it's rolled out everywhere.

**How we'd know it worked:** Compare a group that gets the reminder against a group that doesn't, and see who comes back within a week.


## Recommendation 3 - Hold the WhatsApp campaign budget until we test it fairly

**What to do:** Pause the 5x budget request. Run a proper test first - send the message to half of eligible people at random, not the current approach, and compare fairly.

**Why:** The claimed number is roughly double the real one. Spending 5x on the wrong number is a bigger risk than waiting a few weeks for a fair test.

**Cost / risk:** Almost no cost to wait. The real risk is spending real budget on a number that's likely half right.

**How we'd know it worked:** Run the fair test, see if the benefit holds at around 8.8 points. If it does, that's the number to build the real budget case on.
